# Multimodal J-space transfer pilot — SpokenCOCO x Gemma 4 E4B IT

**One question.** Does a fixed, previously fitted, *text-calibrated* J-lens expose
J-space coordinates that identify the same concept across written text, images,
and spoken captions — and can a concept direction estimated in one modality
causally move that concept in another, with no cross-modal alignment learned
anywhere?

**What SpokenCOCO can and cannot answer.** It carries COCO images, written
captions, and spoken readings of those captions. So this tests transfer among
**visual**, **written-linguistic**, and **spoken-linguistic** evidence. It says
nothing about environmental audio — no barking, no sirens, no instruments. The
modality names used throughout are `text`, `image`, and `spoken_audio`, and the
pair that matters most is text <-> image; speech is an extra condition.

**What this notebook is not.** No phrase generation, no latent-to-language
decoding, no reconstruction, no contrastive alignment, no learned modality
mapping, no supervised probe as the primary method. Interventions add and
subtract a direction on the residual stream — that is not erasure and not
projection ablation.

**What counts as a positive.** A concept is present in a synchronized group
only when **both** kinds of evidence are there: the COCO object annotation for
the image, **and** the concept (or an approved synonym) in the written caption,
matched whole-word. The first real run labelled positives from the annotation
alone; SpokenCOCO images routinely show an object the caption never mentions, so
the text arm was asked about a concept its caption did not state, and failed the
behavioral gate asymmetrically (image 8/8, text 3-6/8). Sections 4-6 audit that
before a single model pass.

**Two switches, both off.** `RUN_REAL_PILOT = False` and
`RUN_MODEL_STAGES = False` by default. Nothing downloads Gemma, mounts Drive, or
reads your dataset until you change them yourself. Sections 4-6 are **CPU only**
and need no GPU: run them first on a free runtime, read the concept ranking, and
stop. Only switch `RUN_MODEL_STAGES = True` on an L4 once the audit reports at
least two feasible concepts. With `RUN_REAL_PILOT` off, the identical cells run
against a deterministic synthetic world (MOCK), which proves the pipeline works
and proves nothing about Gemma.

**Order of stages.** 0 bootstrap, 1 configuration, 2 Drive, 3 dependencies,
**4 SpokenCOCO audit (CPU)**, **5 synchronized-evidence audit (CPU)**,
**6 pilot subset (CPU)**, 7 authentication, 8 model-architecture audit,
9 capability gate, 10 lens validation, 11 activations, 12 J-space codes,
13 representational tests, 14 concept directions, 15 causal transfer,
16 GO/NO-GO report, 17 resume status. Everything from section 7 on is gated on
`RUN_MODEL_STAGES`.

## 0. Colab bootstrap

Run these three cells first, in order. They use nothing but the standard
library: the repository is not importable until 0c has installed it, so
anything that says `from jlens...` before then would fail with
`ModuleNotFoundError: No module named 'jlens'`.

Google Drive is **not** needed here — the package is installed and verified
before section 2 mounts anything.

In [ ]:
# 0a. Bootstrap constants only. Nothing from this repository is imported yet.
REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
BRANCH = "experiment/spokencoco-jspace-pilot"
REPO_DIR = "/content/jacobian-lens-gemma"

print(f"repo   {REPO_URL}")
print(f"branch {BRANCH}")
print(f"target {REPO_DIR}")

In [ ]:
# 0b. Clone or update the repository, then verify the checked-out branch.
#
# Idempotent: clones when absent, otherwise fetches the branch, checks it out,
# and resets to origin. The reset discards local edits inside the Colab
# checkout — that directory is scratch, not somewhere to keep work.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

GITHUB_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata

        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None


def _git(args, *, auth=False, check=True):
    """Run git. Authentication, when present, goes in a per-invocation header
    override so the token never reaches .git/config or the remote URL."""
    prefix = []
    if auth and GITHUB_TOKEN:
        import base64

        encoded = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
        prefix = ["-c", f"http.https://github.com/.extraHeader=AUTHORIZATION: basic {encoded}"]
    result = subprocess.run(["git", *prefix, *args], capture_output=True, text=True)
    if check and result.returncode != 0:
        stderr = result.stderr
        if GITHUB_TOKEN:
            stderr = stderr.replace(GITHUB_TOKEN, "***")
        raise RuntimeError(f"git {' '.join(args)} failed:\n{stderr.strip()}")
    return result.stdout.strip()


def _find_local_checkout() -> Path:
    """Outside Colab, use the checkout this notebook already lives in."""
    override = os.environ.get("MMPILOT_REPO_DIR")
    if override:
        return Path(override).resolve()
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "jlens").is_dir():
            return candidate
    raise RuntimeError(
        "cannot locate the jacobian-lens-gemma checkout. Set MMPILOT_REPO_DIR "
        "to it, or run this notebook from inside the repository."
    )


if IN_COLAB:
    REPO_PATH = Path(REPO_DIR)
    if not (REPO_PATH / ".git").is_dir():
        if REPO_PATH.exists():
            raise RuntimeError(
                f"{REPO_PATH} exists but is not a git checkout; move or remove it "
                "yourself, then re-run this cell."
            )
        print(f"cloning {BRANCH} ...")
        _git(["clone", "--branch", BRANCH, REPO_URL, str(REPO_PATH)], auth=True)
    else:
        print(f"updating existing checkout at {REPO_PATH} ...")
        _git(["-C", str(REPO_PATH), "fetch", "origin", BRANCH], auth=True)
        _git(["-C", str(REPO_PATH), "checkout", BRANCH])
        _git(["-C", str(REPO_PATH), "reset", "--hard", f"origin/{BRANCH}"])
else:
    REPO_PATH = _find_local_checkout()
    print(f"not in Colab — using the existing checkout at {REPO_PATH}")

CHECKED_OUT_BRANCH = _git(["-C", str(REPO_PATH), "rev-parse", "--abbrev-ref", "HEAD"])
COMMIT = _git(["-C", str(REPO_PATH), "rev-parse", "HEAD"])
if IN_COLAB and CHECKED_OUT_BRANCH != BRANCH:
    raise RuntimeError(
        f"expected branch {BRANCH}, but {REPO_PATH} is on {CHECKED_OUT_BRANCH}; "
        "refusing to continue against the wrong code."
    )
print(f"branch: {CHECKED_OUT_BRANCH}")
print(f"commit: {COMMIT}")

In [ ]:
# 0c. Install the repository, move into it, and verify that `import jlens`
# resolves to this checkout. Every later cell may import from the package.
if IN_COLAB:
    print("installing the repository (editable) ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_PATH)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            "pip install -e failed; the package is not available and nothing "
            f"below will import:\n{result.stdout[-1500:]}\n{result.stderr[-2000:]}"
        )
    print(result.stdout.strip().splitlines()[-1] if result.stdout.strip() else "installed")

os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

import importlib

importlib.invalidate_caches()
try:
    import jlens
    import jlens.mmpilot
except ModuleNotFoundError as exc:
    raise RuntimeError(
        f"the repository is still not importable after installation: {exc}. "
        "Re-run cells 0a-0c; do not continue past this point."
    ) from exc

if Path(jlens.__file__).resolve().parent.parent != REPO_PATH:
    raise RuntimeError(
        f"`import jlens` resolved to {jlens.__file__}, which is outside "
        f"{REPO_PATH}. Another copy of the package is shadowing this checkout."
    )
print(f"cwd:         {os.getcwd()}")
print(f"jlens.__file__: {jlens.__file__}")

## 1. Configuration

In [ ]:
# 1. Configuration. Requires section 0 to have run (it imports from the
# repository). Nothing here loads a model, mounts Drive, or reads data.
#
# Flip RUN_REAL_PILOT to True by hand when you want the real experiment. Leave
# it False to exercise every cell below against the deterministic MOCK world.
RUN_REAL_PILOT = False

# RUN_MODEL_STAGES gates everything that needs a GPU: loading Gemma, the
# capability gate, activations, J-space codes, directions, interventions and the
# GO/NO-GO report. Sections 4-6 — metadata expansion, the synchronized-evidence
# audit, concept ranking and the split — run on CPU with this left False, so the
# audit can be done on a free runtime before any L4 is started.
#
# Change this to True BY HAND, and only after the section 6 audit reports at
# least MIN_CONCEPTS_REQUIRED feasible concepts. Nothing flips it for you.
RUN_MODEL_STAGES = False

# TINY_SMOKE validates real Drive reads and media decoding with 2 concepts x 2
# groups. It is NOT scientifically meaningful and never feeds the research verdict.
TINY_SMOKE = False

# The installed Gemma 4 processor/model audio path produced audio features
# but zero audio placeholder tokens. Keep speech explicitly blocked in the
# diagnostic follow-up; no transcript substitution is permitted.
ENABLE_SPOKEN_AUDIO = False

# --- Google Drive locations. The dataset already exists in Drive: it is
# never re-downloaded, never rewritten, and the original manifest is never mutated.
#
# Images and audio do NOT share a root. The observed layout is
#
#   cstf_spokencoco/
#   ├── coco/            train2014/ val2014/     <- manifest says "train2014/....jpg"
#   ├── SpokenCOCO/      wavs/                   <- manifest says "wavs/train/....wav"
#   └── cstf_dataset_marker.json
#
# so an image path only resolves under coco/ and an audio path only under
# SpokenCOCO/. Section 6 audits this before normalizing and prints which root
# resolved which path.
SPOKENCOCO_BASE_ROOT = "/content/drive/MyDrive/datasets/cstf_spokencoco"
IMAGE_MEDIA_ROOT = "/content/drive/MyDrive/datasets/cstf_spokencoco/coco"
AUDIO_MEDIA_ROOT = "/content/drive/MyDrive/datasets/cstf_spokencoco/SpokenCOCO"
DOWNLOAD_CACHE = "/content/drive/MyDrive/datasets/cstf_spokencoco_download_cache"
MANIFEST_PATH = "/content/drive/MyDrive/datasets/spokencoco_manifest.json"
RUNS_ROOT = "/content/drive/MyDrive/jacobian-lens-gemma/runs"

# --- The frozen, text-fitted lens produced by the earlier pilot run.
LENS_RUN_DIR_NAME = "pilot_20260715T200437612150_311fd108c23a"
LENS_ARTIFACT_RELPATH = "artifacts/lens.pt"
LENS_EXPECT_SHA256 = (
    "sha256:7229c7562d1d55420b70abb13f481934649c4b01417bd851e97cedb47c96f474"
)

# --- Model, pinned to the immutable revision the lens was fitted at.
MODEL_REPO_ID = "google/gemma-4-E4B-it"
MODEL_REVISION = "fa62d88df2e6df5efa9d26ad6b3beaea2765f0cd"
EXPECT_N_LAYERS, EXPECT_D_MODEL, EXPECT_VOCAB = 42, 2560, 262144

# --- Scope. Two fitted layers for activations; the deeper one for interventions.
LAYERS = (35, 38)
CAUSAL_LAYERS = (38,)
GROUPS_PER_CONCEPT = 6
NEGATIVES_PER_CONCEPT = 6
N_TARGET_EXAMPLES = 4
ALPHAS = (0.0, 0.25, 0.5)
CAPABILITY_THRESHOLD = 0.7

# --- Candidate concepts and the evidence rule.
#
# CONCEPT_LEXICON maps each concept to the written terms that count as
# expressing it: the word, its plural, and synonyms a careful reader accepts
# without argument (a kitten is a cat). Matching is normalized whole-word regex,
# so 'cat' never matches 'cattle'. No embeddings, no classifier, no fuzzy
# similarity, nothing learned. Edit it here if you want to inspect a different
# set — the lexicon is hashed into every derived artifact.
from jlens.mmpilot.evidence import CONCEPT_LEXICON, config_for_concepts

CONCEPT_CANDIDATES = dict(CONCEPT_LEXICON)
EVIDENCE_CONFIG = config_for_concepts(CONCEPT_CANDIDATES)

# Screen up to four concepts; refuse to go past the behavioral gate with fewer
# than two. These are NOT lowered automatically — a smaller split would change
# what a GO verdict means.
N_CONCEPTS_TO_SCREEN = 4
MIN_CONCEPTS_REQUIRED = 2

# --- Manifest field overrides. Leave empty: section 6 resolves the schema by
# inspection and refuses if it is ambiguous. Fill in only what it asks for,
# e.g. {"caption": "text", "audio": "wav"}.
MANIFEST_OVERRIDES: dict = {}

if TINY_SMOKE and not RUN_REAL_PILOT:
    raise RuntimeError("TINY_SMOKE=True requires RUN_REAL_PILOT=True (plumbing on real media only)")
print(f"RUN_REAL_PILOT   = {RUN_REAL_PILOT}")
print(f"RUN_MODEL_STAGES = {RUN_MODEL_STAGES}")
print(f"TINY_SMOKE       = {TINY_SMOKE}")
print(f"evidence lexicon hash: {EVIDENCE_CONFIG.lexicon_hash}")
for _name in sorted(CONCEPT_CANDIDATES):
    print(f"  {_name:8s} terms={list(CONCEPT_CANDIDATES[_name])} "
          f"coco={list(EVIDENCE_CONFIG.categories_for(_name))}")
if not RUN_MODEL_STAGES:
    print("\nMODEL STAGES ARE OFF. Sections 4-6 run on CPU; sections 7-17 are skipped.")
if TINY_SMOKE:
    print("MODE = TINY_SMOKE (plumbing validation only — NOT a scientific pilot)")
elif RUN_REAL_PILOT:
    print("MODE = PILOT (real Gemma, real SpokenCOCO)")
else:
    print("MODE = MOCK (synthetic)")

## 2. Mount Google Drive

In [ ]:
# 2. Mount Drive and verify the configured paths exist. Read-only checks: this
# cell never creates, moves, or deletes anything inside the dataset.
#
# `os`, `sys`, `Path` and `IN_COLAB` come from the bootstrap in section 0.
import tempfile

if RUN_REAL_PILOT and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

if RUN_REAL_PILOT:
    missing = [
        path
        for path in (SPOKENCOCO_BASE_ROOT, MANIFEST_PATH)
        if not (Path(path).is_dir() or Path(path).is_file())
    ]
    if missing:
        raise RuntimeError(
            f"configured Drive paths not found: {missing}. Fix the paths in "
            "section 1; nothing is re-downloaded by this notebook."
        )
    # Every root, in priority order, offered to both modalities: the image root
    # first, then the audio root, then the cache, then the base root as a
    # catch-all for layouts that do keep everything together. Roots that do not
    # exist are dropped, and section 6 reports which one actually resolved what.
    CANDIDATE_ROOTS = [
        IMAGE_MEDIA_ROOT,
        AUDIO_MEDIA_ROOT,
        DOWNLOAD_CACHE,
        SPOKENCOCO_BASE_ROOT,
    ]
    IMAGE_ROOTS = [Path(root) for root in CANDIDATE_ROOTS if Path(root).is_dir()]
    AUDIO_ROOTS = list(IMAGE_ROOTS)
    EXPECTED_ROOTS = {"image": IMAGE_MEDIA_ROOT, "audio": AUDIO_MEDIA_ROOT}
    RESOLVED_RUNS_ROOT = Path(RUNS_ROOT)
    print(f"base root:   {SPOKENCOCO_BASE_ROOT}")
    print(f"image root:  {IMAGE_MEDIA_ROOT}  exists={Path(IMAGE_MEDIA_ROOT).is_dir()}")
    print(f"audio root:  {AUDIO_MEDIA_ROOT}  exists={Path(AUDIO_MEDIA_ROOT).is_dir()}")
    print(f"cache:       {DOWNLOAD_CACHE}  exists={Path(DOWNLOAD_CACHE).is_dir()}")
    print(f"manifest:    {MANIFEST_PATH}")
    print(f"roots offered (priority order): {[str(root) for root in IMAGE_ROOTS]}")
    for skipped in CANDIDATE_ROOTS:
        if not Path(skipped).is_dir():
            print(f"  (skipping {skipped} — not a directory)")
else:
    SCRATCH = Path(os.environ.get("MMPILOT_SCRATCH") or tempfile.mkdtemp(prefix="mmpilot_"))
    # The MOCK dataset keeps both modalities under one root — the layout the
    # per-role roots must stay backward compatible with.
    IMAGE_ROOTS = [SCRATCH / "data" / "coco", SCRATCH / "data" / "SpokenCOCO", SCRATCH / "data"]
    AUDIO_ROOTS = list(IMAGE_ROOTS)
    EXPECTED_ROOTS = None
    RESOLVED_RUNS_ROOT = SCRATCH / "runs"
    print(f"MOCK mode — Drive is not mounted. Scratch: {SCRATCH}")

MEDIA_ROOTS = list(IMAGE_ROOTS)  # kept for display; resolution is per role

RESOLVED_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print(f"runs root:    {RESOLVED_RUNS_ROOT}")

## 3. Install dependencies

In [ ]:
# 3. Media dependencies (the repository itself was installed in section 0c) and
# the runtime report. Never touches the dataset or the model.
if IN_COLAB:
    command = [sys.executable, "-m", "pip", "install", "-q",
               "pillow", "soundfile", "librosa"]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{result.stderr[-2000:]}")
    print("media dependencies installed")
else:
    print("not in Colab — media dependencies are only needed for the real pilot")

import platform

import torch

print(f"python {platform.python_version()}  torch {torch.__version__}  cuda {torch.version.cuda}")
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_properties(0).name,
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

## 4. Audit SpokenCOCO (CPU only)

The manifest's schema is **discovered, not assumed**: candidate fields are
scored by their names *and* by whether their values actually resolve to files on
disk, and an ambiguous mapping is refused rather than guessed.

A media-root audit runs **before** normalization. Images and audio do not share
a root in this dataset, so each modality is resolved against its own ordered
list of roots, and the audit prints which root resolved each sampled path. It
fails loudly when nothing resolves for a modality, when a path matches several
roots holding different files, or when the image and audio roots look swapped.

The audit then reports record counts, valid media, synchronized groups, missing
files, duplicates, splits, and speaker metadata, and writes a normalized derived
manifest into the run directory next to the original's checksum. The original is
never modified.

If fuller SpokenCOCO annotation files (for example ``SpokenCOCO_train.json``) are
already on disk under ``coco/`` or ``SpokenCOCO/``, a separate **expanded**
manifest is derived from them with the same synchronization rules. Nothing is
re-downloaded; a group exists only when its image and recording are both present
locally. Scientific coverage thresholds are never lowered automatically — if the
local data cannot support two concepts at ``GROUPS_PER_CONCEPT=6``, section 6
stops with an explicit DATASET NO-GO.

**No model is loaded in this section.** It runs on a free CPU runtime.

In [ ]:
# 4. CPU ONLY. Inspect, audit and normalize the original manifest, then expand
# from fuller local metadata already on disk (read-only throughout). No model is
# loaded here, so this whole section runs on a free CPU runtime.
import json
from datetime import datetime, timezone

from jlens.mmpilot import expansion as expansion_module
from jlens.mmpilot import manifest as manifest_module
from jlens.mmpilot.pipeline import PilotConfig

# The run's identity and scope, needed before any data is read. Building it here
# rather than beside the model keeps the audit independent of a GPU.
CONFIG = PilotConfig(
    mode=("tiny_smoke" if TINY_SMOKE else ("pilot" if RUN_REAL_PILOT else "mock")),
    layers=tuple(LAYERS) if RUN_REAL_PILOT else (2, 4),
    causal_layers=tuple(CAUSAL_LAYERS) if RUN_REAL_PILOT else (4,),
    capability_threshold=CAPABILITY_THRESHOLD,
    alphas=tuple(ALPHAS),
    n_target_examples=N_TARGET_EXAMPLES,
    pursuit_k=25 if RUN_REAL_PILOT else 8,
    pursuit_correlation_chunk_size=65536 if RUN_REAL_PILOT else None,
    direction_top_k=16 if RUN_REAL_PILOT else 4,
)

if not RUN_REAL_PILOT:
    # The synthetic world. `visual_only_images` plants the exact failure mode
    # this repair is about: images annotated with a concept whose captions never
    # name it. They must be rejected as synchronized positives.
    from jlens.mmpilot.mock import build_mock_dataset

    if not (SCRATCH / "data" / "spokencoco_manifest.json").is_file():
        build_mock_dataset(
            SCRATCH / "data",
            layout="sibling",
            manifest_records=8,
            visual_only_images=2,
        )
    MANIFEST_PATH = str(SCRATCH / "data" / "spokencoco_manifest.json")
    print(f"MOCK dataset ready at {SCRATCH / 'data'}")

MANIFEST_PAYLOAD = json.loads(Path(MANIFEST_PATH).read_text(encoding="utf-8"))
MANIFEST_CHECKSUM = manifest_module.manifest_checksum(MANIFEST_PATH)
SCHEMA = manifest_module.inspect_manifest(
    MANIFEST_PAYLOAD, overrides=MANIFEST_OVERRIDES or None
)
print("resolved schema:")
print(json.dumps(SCHEMA.to_dict(), indent=2, default=str)[:2000])

# Media-root audit BEFORE normalization: probe a handful of representative
# paths and print which root resolves each, so a root misconfiguration is
# diagnosed against 8 paths rather than as "96 of 96 did not resolve".
MEDIA_ROOT_CONFIG = manifest_module.resolve_media_roots(
    image_roots=IMAGE_ROOTS, audio_roots=AUDIO_ROOTS
)
MEDIA_ROOT_AUDIT = manifest_module.audit_media_roots(
    MANIFEST_PAYLOAD, SCHEMA, MEDIA_ROOT_CONFIG, expected_roots=EXPECTED_ROOTS
)
print("\nmedia-root audit:")
for role in ("image", "audio"):
    print(f"  {role}:")
    for sample in MEDIA_ROOT_AUDIT["samples"][role][:4]:
        print(f"    {sample['relative']}")
        print(f"      -> {sample['resolved']}")
        print(f"         via {sample['root']} ({sample['mode']})")
    print(f"    resolved by root: {MEDIA_ROOT_AUDIT['resolution_by_root'][role]}")
if MEDIA_ROOT_AUDIT["transient_io_retries"]:
    print(
        f"\n  note: {MEDIA_ROOT_AUDIT['transient_io_retries']} transient Drive I/O "
        "error(s) were retried and cleared. Drive is being flaky; if a later "
        "stage stops with a MediaIOError, remount and re-run — the run "
        "directory resumes."
    )

BASELINE_NORMALIZED = manifest_module.normalize_manifest(
    MANIFEST_PAYLOAD,
    SCHEMA,
    image_roots=IMAGE_ROOTS,
    audio_roots=AUDIO_ROOTS,
    source_checksum=MANIFEST_CHECKSUM,
    min_complete_groups=1,
)
print("\nbaseline audit (original manifest only):")
print(json.dumps(BASELINE_NORMALIZED.audit, indent=2, default=str))
# Ranked before expansion, under the same evidence rule. The original manifest
# carries no COCO object annotations, so nothing in it can be a valid
# synchronized positive on its own — expansion below is what supplies the visual
# half of the rule.
BASELINE_RANKING = expansion_module.rank_concepts(
    BASELINE_NORMALIZED.groups,
    CONCEPT_CANDIDATES,
    groups_per_concept=GROUPS_PER_CONCEPT,
    evidence_config=EVIDENCE_CONFIG,
)
BASELINE_COVERAGE_SUFFICIENT = (
    sum(row["feasible"] for row in BASELINE_RANKING) >= MIN_CONCEPTS_REQUIRED
)
print("\noriginal-manifest concept coverage:")
print(expansion_module.format_ranking_table(BASELINE_RANKING))
print(f"coverage sufficient in original manifest? {BASELINE_COVERAGE_SUFFICIENT}")

RUN_ID = (
    f"mmpilot_{CONFIG.mode}_"
    f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
)
RUN_DIR = Path(os.environ.get("MMPILOT_RUN_DIR") or (RESOLVED_RUNS_ROOT / RUN_ID))
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "derived_manifest.json").write_text(
    json.dumps(BASELINE_NORMALIZED.to_dict(), indent=2, default=str), encoding="utf-8"
)
print(f"\nderived manifest (original, unchanged on disk) -> {RUN_DIR / 'derived_manifest.json'}")
print(f"original manifest checksum: {MANIFEST_CHECKSUM}")

SEARCH_ROOTS = []
if RUN_REAL_PILOT:
    for candidate in (
        SPOKENCOCO_BASE_ROOT,
        IMAGE_MEDIA_ROOT,
        AUDIO_MEDIA_ROOT,
        DOWNLOAD_CACHE,
    ):
        if Path(candidate).is_dir():
            SEARCH_ROOTS.append(candidate)
else:
    SEARCH_ROOTS = sorted({str(root) for root in IMAGE_ROOTS if root.is_dir()})

if BASELINE_COVERAGE_SUFFICIENT:
    DISCOVERED = []
    print("original manifest already meets the scientific coverage gate; metadata expansion skipped")
else:
    DISCOVERED = expansion_module.discover_metadata_sources(
        SEARCH_ROOTS, exclude=[MANIFEST_PATH], max_files=40, max_depth=3,
    )
print("\nlocal metadata sources (largest first):")
for source in DISCOVERED:
    print(
        f"  {source.path}\n"
        f"    size={source.size_bytes} format={source.detected_format} depth={source.recursion_depth}\n"
        f"    kind={source.source_kind} usable={source.usable} records={source.n_records}\n"
        f"    checksum={source.checksum or 'n/a'}\n"
        f"    top-level schema={source.top_level_schema}\n"
        f"    likely fields={source.likely_fields}\n"
        f"    decision={source.reason}"
    )

EXPANSION = expansion_module.build_expanded_manifest(
    [source for source in DISCOVERED if source.usable],
    image_roots=IMAGE_ROOTS,
    annotation_sources=[source for source in DISCOVERED if source.source_kind == "coco_object_annotation"],
    candidate_concepts=CONCEPT_CANDIDATES,
    max_metadata_records=20000,
    audio_roots=AUDIO_ROOTS,
    baseline_groups=BASELINE_NORMALIZED.groups,
)
PILOT_GROUPS = EXPANSION.groups
EXPANDED_MANIFEST_PATH = RUN_DIR / "expanded_manifest.json"
CONVERSION_CONFIG = {
    "converter": "jlens.mmpilot.expansion.build_expanded_manifest",
    "search_roots": SEARCH_ROOTS, "max_depth": 3, "max_files": 40, "max_metadata_records": 20000,
    "groups_per_concept": GROUPS_PER_CONCEPT,
    "n_concepts_screened": N_CONCEPTS_TO_SCREEN,
    "min_concepts_required": MIN_CONCEPTS_REQUIRED,
    "evidence_rule": "visual_annotation_AND_caption_lexicon",
    "evidence_lexicon_hash": EVIDENCE_CONFIG.lexicon_hash,
    "evidence_config_fingerprint": EVIDENCE_CONFIG.fingerprint,
    "reads_only": True, "media_redownloaded": False, "audio_transcribed": False,
}
EXPANSION_PAYLOAD, EXPANSION_STATUS = expansion_module.persist_expanded_manifest(
    EXPANDED_MANIFEST_PATH, EXPANSION, original_checksum=MANIFEST_CHECKSUM, conversion=CONVERSION_CONFIG
)
print(EXPANSION_STATUS)
print(
    f"\nexpanded manifest: {EXPANSION.n_groups} synchronized groups "
    f"({len(EXPANSION.baseline_group_ids)} from the original manifest, "
    f"{EXPANSION.n_groups - len(EXPANSION.baseline_group_ids)} added from local metadata)"
)
FINAL_MANIFEST_KIND = "expanded_derived" if EXPANSION.n_groups > len(EXPANSION.baseline_group_ids) else "original"
FINAL_MANIFEST_PATH = EXPANDED_MANIFEST_PATH if FINAL_MANIFEST_KIND == "expanded_derived" else Path(MANIFEST_PATH)
print(f"final manifest: {FINAL_MANIFEST_KIND} -> {FINAL_MANIFEST_PATH}")

NORMALIZED = BASELINE_NORMALIZED
MANIFEST_AUDIT = {
    **BASELINE_NORMALIZED.audit,
    "n_expanded_groups": len(PILOT_GROUPS),
    "n_baseline_groups": len(BASELINE_NORMALIZED.groups),
    "expansion_per_source": EXPANSION.per_source,
}

# The visual half of the evidence rule comes from COCO object annotations. If
# none were found, no group can be a valid synchronized positive — say so here
# rather than letting section 6 report an unexplained zero.
ANNOTATION_SOURCES = [
    source for source in DISCOVERED if source.source_kind == "coco_object_annotation"
]
N_ANNOTATED_GROUPS = sum(1 for g in PILOT_GROUPS if g.get("concept_annotations"))
print(f"\nCOCO object-annotation files found: {[s.path for s in ANNOTATION_SOURCES] or 'NONE'}")
print(f"groups carrying a COCO object annotation: {N_ANNOTATED_GROUPS} / {len(PILOT_GROUPS)}")
if not N_ANNOTATED_GROUPS:
    print(
        "\nWARNING: no group carries visual annotation evidence. A valid "
        "synchronized positive needs BOTH the COCO object annotation and the "
        "caption term, so section 6 will report a DATASET NO-GO. Put COCO "
        "instances_*.json (the object annotations, not captions_*.json) under "
        "the image root's annotations/ directory and re-run."
    )

## 5. Audit synchronized evidence (CPU only)

This is the section the first real run needed and did not have.

**The rule.** A group is a valid synchronized positive for a concept only when
it carries **both**:

1. **visual evidence** — a COCO object annotation for that concept on its image;
   and
2. **caption evidence** — the concept, or an approved synonym from the explicit
   lexicon, present in the written caption as a whole word or phrase.

**Why both.** The earlier rule short-circuited on the annotation and never read
the caption. SpokenCOCO images routinely contain an object the caption does not
mention, so a group could be labelled a `cat` positive while its caption said
"a fluffy animal asleep on a sofa". The image arm then answered correctly and
the text arm was scored wrong for answering honestly — image 8/8, text 3-6/8.
A group like that holds visual evidence and no written evidence; the two
modalities are not carrying the same claim, so it is not a valid test of
transfer.

**Matching.** Normalized whole-word regex against the lexicon printed in section
1. `cat` does not match `cattle`; `bus` does not match `business`. No
embeddings, no learned classifier, no language model, no fuzzy similarity, no
network call. Every match records the normalized caption and the matched span so
it can be re-checked by hand.

**Speech.** SpokenCOCO recordings are spoken readings of the written captions,
so caption evidence describes what linguistic content the recording carries.
That is a fact about the dataset's metadata. **No audio is transcribed**, and
none of this is evidence that Gemma can hear.

**Negatives.** A matched negative must carry *neither* kind of evidence for
*any* screened concept. A visual-only picture of a cat is excluded from the
negatives as well as from the positives — contrasting against it would blunt the
direction the contrast is meant to isolate.

Artifacts written into the run directory (the originals are never touched):
`synchronized_evidence_audit.json`, `synchronized_evidence_manifest.json`,
`concept_ranking.json`. Each is written atomically, carries the lexicon and
configuration hashes plus the source checksums, and refuses to be reused under a
different configuration.

In [ ]:
# 5. CPU ONLY. The synchronized-evidence audit. No model, no GPU, no transcript.
from jlens.mmpilot import evidence as evidence_module

EVIDENCE_SOURCE_CHECKSUMS = evidence_module.source_checksums(
    [MANIFEST_PATH, *[source.path for source in ANNOTATION_SOURCES]]
)
EVIDENCE_AUDIT = evidence_module.audit_groups(
    PILOT_GROUPS,
    config=EVIDENCE_CONFIG,
    concepts=sorted(CONCEPT_CANDIDATES),
    source_checksums=EVIDENCE_SOURCE_CHECKSUMS,
)
EVIDENCE_CONVERSION = {
    **CONVERSION_CONFIG,
    "stage": "synchronized_evidence_audit",
    "concepts_screened": sorted(CONCEPT_CANDIDATES),
}

AUDIT_PAYLOAD, AUDIT_STATUS = evidence_module.persist_evidence_audit(
    RUN_DIR / "synchronized_evidence_audit.json",
    EVIDENCE_AUDIT,
    conversion=EVIDENCE_CONVERSION,
)
SYNC_MANIFEST_PAYLOAD, SYNC_MANIFEST_STATUS = evidence_module.persist_synchronized_manifest(
    RUN_DIR / "synchronized_evidence_manifest.json",
    PILOT_GROUPS,
    EVIDENCE_AUDIT,
    original_checksum=MANIFEST_CHECKSUM,
    conversion=EVIDENCE_CONVERSION,
)
print(AUDIT_STATUS)
print(SYNC_MANIFEST_STATUS)
print(f"\nevidence rule: visual COCO annotation AND caption lexicon (both required)")
print(f"lexicon hash:  {EVIDENCE_CONFIG.lexicon_hash}")
print(f"config hash:   {EVIDENCE_CONFIG.fingerprint}")
print(f"source checksums: {json.dumps(EVIDENCE_SOURCE_CHECKSUMS, indent=2)}")

print(
    f"\ngroups examined: {EVIDENCE_AUDIT.n_groups}   "
    f"candidate positives: {len(EVIDENCE_AUDIT.records)}   "
    f"valid synchronized positives: {len(EVIDENCE_AUDIT.valid_records())}   "
    f"candidate negatives: {len(EVIDENCE_AUDIT.negatives)}"
)
print("\nrejection counts by reason:")
print(evidence_module.format_rejection_counts(EVIDENCE_AUDIT))

print("\nper-concept rejection counts:")
for _concept, _counts in sorted(EVIDENCE_AUDIT.rejection_counts_by_concept().items()):
    print(f"  {_concept:10s} " + "  ".join(
        f"{_reason}={_n}" for _reason, _n in sorted(_counts.items()) if _n
    ))

# Re-rank on valid synchronized positives only. 'annot' and 'capt' are the two
# halves of the rule; 'images' and 'groups' count only what satisfies both, so
# annot >> images is exactly the failure this repair diagnoses.
RANKING = expansion_module.rank_concepts(
    PILOT_GROUPS,
    CONCEPT_CANDIDATES,
    requirements=(
        expansion_module.tiny_smoke_requirements()
        if TINY_SMOKE
        else expansion_module.ConceptRequirements()
    ),
    groups_per_concept=2 if TINY_SMOKE else GROUPS_PER_CONCEPT,
    evidence_config=EVIDENCE_CONFIG,
)
print("\nconcept ranking (valid synchronized positives only):")
print(expansion_module.format_ranking_table(RANKING))

RANKING_PAYLOAD, RANKING_STATUS = evidence_module.persist_concept_ranking(
    RUN_DIR / "concept_ranking.json",
    RANKING,
    EVIDENCE_AUDIT,
    requirements=(
        expansion_module.tiny_smoke_requirements()
        if TINY_SMOKE
        else expansion_module.ConceptRequirements()
    ).to_dict(),
    conversion=EVIDENCE_CONVERSION,
)
print(RANKING_STATUS)

# Representative accepted and rejected groups, bounded so this is readable.
# Read these before spending anything on a GPU: they are how you check that the
# lexicon is doing what you think it is.
print("\n" + "=" * 72)
print("REPRESENTATIVE EXAMPLES — verify these by eye before any model run")
print("=" * 72)
print(evidence_module.format_examples(
    EVIDENCE_AUDIT, sorted(CONCEPT_CANDIDATES), n_positive=3, n_rejected=3
))
print("\nrepresentative matched negatives (no evidence of any kind):")
for _negative in EVIDENCE_AUDIT.negatives[:3]:
    print(f"  group {_negative['group_id']}  image {_negative['image_id']}")
    print(f"    caption: {' '.join(_negative['caption'].split())[:88]}")
    print(f"    qualifies: {_negative['qualifies_as_negative']} "
          f"({_negative['qualification_rule']})")

## 6. Build the pilot subset (CPU only)

Selection uses **only** valid synchronized positives. Thresholds are the stated
scientific ones and are never lowered automatically: 6 distinct images, 6
synchronized groups, 4 source-training positives, 2 held-out positives, and 6
matched negatives per concept. Up to four concepts are screened; fewer than two
feasible is a DATASET NO-GO.

Every group belonging to one COCO image lands in the same split, so all captions
and recordings of an image stay together and the split is image-, group-,
caption- and audio-disjoint.

This is the last CPU-only section. If `RUN_MODEL_STAGES` is False, the notebook
stops reporting here and nothing below runs.

In [ ]:
# 6. CPU ONLY. Select concepts from the ranking above and build the split.
PILOT_REQUIREMENTS = (
    expansion_module.tiny_smoke_requirements()
    if TINY_SMOKE
    else expansion_module.ConceptRequirements()
)
GROUPS_FOR_SPLIT = 2 if TINY_SMOKE else GROUPS_PER_CONCEPT
NEGATIVES_FOR_SPLIT = 2 if TINY_SMOKE else NEGATIVES_PER_CONCEPT

SELECTED_NAMES = expansion_module.select_concepts(
    RANKING,
    n_concepts=2 if TINY_SMOKE else MIN_CONCEPTS_REQUIRED,
    max_concepts=2 if TINY_SMOKE else N_CONCEPTS_TO_SCREEN,
    total_synchronized_records=len(PILOT_GROUPS),
    coverage_cause="local metadata, media validation and synchronized-evidence coverage",
    requirements=PILOT_REQUIREMENTS,
)
SELECTED_CONCEPTS = {name: CONCEPT_CANDIDATES[name] for name in SELECTED_NAMES}
print("selected concepts:", sorted(SELECTED_CONCEPTS))
if TINY_SMOKE:
    print("\n*** TINY_SMOKE: plumbing validation only — NOT scientifically meaningful ***")

SUBSET = manifest_module.build_subset(
    PILOT_GROUPS,
    SELECTED_CONCEPTS,
    groups_per_concept=GROUPS_FOR_SPLIT,
    negatives_per_concept=NEGATIVES_FOR_SPLIT,
    evidence_config=EVIDENCE_CONFIG,
)
LEAKAGE = manifest_module.check_split_leakage(SUBSET)
print("\nsplit check:", json.dumps(LEAKAGE, indent=2, default=str))
if not LEAKAGE["ok"]:
    raise RuntimeError("split leakage detected; refusing to continue")

CONFIG.concepts = tuple(sorted(SELECTED_CONCEPTS))
print(f"\ntrain groups: {LEAKAGE['n_train_groups']}  test groups: {LEAKAGE['n_test_groups']}")
print("speakers in the subset:",
      sorted({g["speaker"] for g in SUBSET["splits"]["train"] if g["speaker"]})[:8])

# Every selected positive, with the evidence that justified it. Read this before
# starting a GPU run: if a caption here does not state its concept, the text arm
# of the behavioral gate is being asked an unanswerable question.
print("\nselected positives and their evidence:")
for _split in ("train", "test"):
    for _row in SUBSET["splits"][_split]:
        if not _row["is_positive"]:
            continue
        _ev = _row["evidence"]
        print(f"  [{_split:5s}] {_row['concept']:8s} group {_row['group_id']}")
        print(f"      caption: {' '.join(_row['caption'].split())[:88]}")
        print(f"      visual:  {_ev['visual_evidence']['matched_categories']}   "
              f"term: {_ev['matched_term']!r} at {_ev['match_span']}")

SPLIT_PROVENANCE = {
    "concepts": SUBSET["concepts"],
    "negatives": SUBSET["negatives"],
    "provenance": SUBSET["provenance"],
    "leakage": LEAKAGE,
}
(RUN_DIR / "split_provenance.json").write_text(
    json.dumps(SPLIT_PROVENANCE, indent=2, default=str), encoding="utf-8"
)
print(f"\nwritten: {RUN_DIR / 'split_provenance.json'}")

print("\n" + "=" * 72)
if RUN_MODEL_STAGES:
    print("CPU AUDIT COMPLETE — RUN_MODEL_STAGES is True, continuing to the model.")
else:
    print("CPU AUDIT COMPLETE. RUN_MODEL_STAGES is False, so sections 7-17 are skipped.")
    print(
        f"\n{len(SELECTED_NAMES)} concept(s) cleared the thresholds: "
        f"{sorted(SELECTED_NAMES)}.\n"
        "Review the ranking table and the representative examples above. If the "
        "captions really do state their concepts, switch to an L4 and set:\n"
        "    RUN_REAL_PILOT    = True\n"
        "    RUN_MODEL_STAGES  = True\n"
        "    TINY_SMOKE        = False\n"
        "    ENABLE_SPOKEN_AUDIO = False\n"
        "Nothing below this cell has run, and no model has been loaded."
    )
print("=" * 72)

## 7. Authenticate

In [ ]:
# 7. HuggingFace authentication (gated Gemma repo). Only needed once the model
# stages are switched on. The token is read with getpass and never printed or
# written to disk by this notebook. The CPU audit above never asks for it.
import getpass

if RUN_REAL_PILOT and RUN_MODEL_STAGES and not os.environ.get("HF_TOKEN"):
    _token = getpass.getpass("HF_TOKEN (input hidden): ").strip()
    if not _token:
        raise RuntimeError("an HF token is required to load the gated Gemma repo")
    os.environ["HF_TOKEN"] = _token
    print(f"HF_TOKEN set ({len(_token)} chars, prefix {_token[:3]}...)")
    del _token
elif RUN_REAL_PILOT and RUN_MODEL_STAGES:
    print("HF_TOKEN already present in the environment")
elif not RUN_MODEL_STAGES:
    print("model stages are off — no authentication needed for the CPU audit")
else:
    print("MOCK mode — no authentication needed")

## 8. Audit the model architecture

**Nothing below this point runs unless `RUN_MODEL_STAGES` is True.**

Loads the model and processor and then **asks them what they support** rather
than trusting names or documentation. If the processor has no audio pathway,
the spoken-audio condition is marked blocked here and stays blocked — speech is
never replaced by its transcript.

Two invariance checks run before anything else touches activations: a capture
hook must leave logits unchanged, and a coefficient-zero edit must reproduce the
clean logits. Either failure aborts the run.

In [ ]:
# 8. Load the model + processor, resolve the interface by inspection, and run
# the two mandatory invariance checks. GATED: nothing here executes while
# RUN_MODEL_STAGES is False, so the CPU audit above never touches a GPU.
from jlens.mmpilot.backend import (
    GemmaPilotBackend,
    resolve_processor_interface,
    run_invariance_gate,
)
from jlens.mmpilot.capability import build_prompt, build_question
from jlens.mmpilot.pipeline import PilotConfig, available_modalities

MODEL = LOAD_INFO = ARCH_REPORT = PROCESSOR = INTERFACE = BACKEND = None
AVAILABLE_MODALITIES = BLOCKED_MODALITIES = []
INVARIANCE = SUMMARY = STATUS = None

if not RUN_MODEL_STAGES:
    print(
        "RUN_MODEL_STAGES is False — no model is loaded and sections 8-17 do "
        "nothing. Set it to True by hand to run the experiment."
    )
elif RUN_REAL_PILOT:
    from transformers import AutoProcessor

    from jlens.gemma4 import load_gemma4, verify_architecture

    MODEL, LOAD_INFO = load_gemma4(
        MODEL_REPO_ID,
        revision=MODEL_REVISION,
        dtype=torch.bfloat16,
        device_map="cuda" if torch.cuda.is_available() else None,
        allow_model_load=True,
    )
    for parameter in MODEL._hf_model.parameters():
        parameter.requires_grad_(False)
    ARCH_REPORT = verify_architecture(
        MODEL,
        expect_n_layers=EXPECT_N_LAYERS,
        expect_d_model=EXPECT_D_MODEL,
        expect_vocab_size=EXPECT_VOCAB,
    ).to_dict()
    PROCESSOR = AutoProcessor.from_pretrained(
        MODEL_REPO_ID, revision=LOAD_INFO["model_revision"]
    )
    INTERFACE = resolve_processor_interface(PROCESSOR, MODEL._hf_model.config)
    BACKEND = GemmaPilotBackend(
        MODEL._hf_model,
        PROCESSOR,
        INTERFACE,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    MODEL_REVISION_USED = LOAD_INFO["model_revision"]
    PROCESSOR_REVISION_USED = LOAD_INFO["model_revision"]
    print(json.dumps(INTERFACE, indent=2, default=str))
    print(json.dumps(
        {k: ARCH_REPORT[k] for k in ("n_layers", "d_model", "vocab_size", "layout_path")},
        indent=2,
    ))
else:
    from jlens.mmpilot.mock import MockPilotBackend

    BACKEND = MockPilotBackend()
    INTERFACE = BACKEND.interface
    MODEL_REVISION_USED = PROCESSOR_REVISION_USED = "mock-rev"
    print("MOCK backend ready:", json.dumps(INTERFACE, indent=2, default=str))

if RUN_MODEL_STAGES:
    AVAILABLE_MODALITIES, BLOCKED_MODALITIES = available_modalities(BACKEND, CONFIG)
if RUN_MODEL_STAGES and RUN_REAL_PILOT and not ENABLE_SPOKEN_AUDIO:
    AVAILABLE_MODALITIES = [m for m in AVAILABLE_MODALITIES if m != 'spoken_audio']
    if 'spoken_audio' not in BLOCKED_MODALITIES:
        BLOCKED_MODALITIES.append('spoken_audio')
    AUDIO_BLOCK_REASON = (
        'processor/model produced audio features but zero audio placeholder tokens; '
        'spoken_audio disabled without transcript substitution'
    )
    print('spoken_audio blocked:', AUDIO_BLOCK_REASON)
print("available modalities:", AVAILABLE_MODALITIES)
print("blocked modalities:  ", BLOCKED_MODALITIES or "none")
if RUN_MODEL_STAGES and "spoken_audio" in BLOCKED_MODALITIES:
    print(
        "\nSPOKEN AUDIO IS BLOCKED. The text-image pilot continues; audio is a "
        "NO-GO for this checkpoint/processor combination. Reason recorded above "
        "in the resolved interface (audio_kwarg / feature_extractor)."
    )

if RUN_MODEL_STAGES:
    _probe = BACKEND.build_inputs(
        prompt=build_prompt(build_question(sorted(CONCEPT_CANDIDATES)), modality="text",
                            caption="a photo of a dog on a bench"),
        modality="text",
    )
    INVARIANCE = run_invariance_gate(BACKEND, _probe, list(CONFIG.layers))
    print("\ninvariance gate:", json.dumps(INVARIANCE, indent=2, default=str))

## 9. Run the capability gate

In [ ]:
# 9. Behavioral gate: same question, same candidates, only the evidence channel
# changes. Complete candidate token sequences are scored by teacher-forced
# conditional log likelihood — never the first token alone.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.mmpilot.pipeline import stage_capability
    from jlens.mmpilot.store import RunFingerprint, UnitStore

    if RUN_REAL_PILOT:
        from PIL import Image

        def _load_image(path):
            return Image.open(path).convert("RGB")

        def _load_audio(path):
            import soundfile as sf

            waveform, sample_rate = sf.read(path, dtype="float32")
            if waveform.ndim > 1:
                waveform = waveform.mean(axis=1)
            return waveform, int(sample_rate)
    else:
        from jlens.mmpilot.mock import load_mock_media

        def _load_image(path):
            return load_mock_media(path)

        def _load_audio(path):
            return load_mock_media(path), 16000

    MEDIA = {"load_image": _load_image, "load_audio": _load_audio}

    FINGERPRINT = RunFingerprint(
        mode=CONFIG.mode,
        model_repo_id=MODEL_REPO_ID if RUN_REAL_PILOT else "mock/gemma-like",
        model_revision=MODEL_REVISION_USED,
        processor_revision=PROCESSOR_REVISION_USED,
        layers=tuple(CONFIG.layers),
        lens_checksum=LENS_EXPECT_SHA256 if RUN_REAL_PILOT else "sha256:mock-identity-lens",
        manifest_checksum=MANIFEST_CHECKSUM,
        split_id=SUBSET["provenance"]["seed"],
        intervention_config={
            "alphas": list(CONFIG.alphas),
            "direction_top_k": CONFIG.direction_top_k,
            "causal_layers": list(CONFIG.causal_layers),
        },
    )
    STORE = UnitStore(RUN_DIR, FINGERPRINT)
    print("run state:", STORE.open())

    CAPABILITY_OUTCOME, CAPABILITY = stage_capability(
        BACKEND, STORE, SUBSET, CONFIG, MEDIA, modalities=AVAILABLE_MODALITIES
    )
    print(CAPABILITY_OUTCOME.line("capability"))

    # With 4-8 samples a percentage is unstable, so print the raw calls too.
    for record in sorted(CAPABILITY_OUTCOME.records, key=lambda r: (r["concept"], r["modality"])):
        print(f"  {record['concept']:10s} {record['modality']:13s} "
              f"pred={record['prediction']:10s} correct={str(record['correct']):5s} "
              f"margin={record['target_margin']:+.3f}")

    print("\nper-concept accuracy:")
    for concept, per_modality in sorted(CAPABILITY["per_concept"].items()):
        print(f"  {concept}: " + "  ".join(
            f"{modality}={entry['n_correct']}/{entry['n']}" for modality, entry in sorted(per_modality.items())
        ))
    RETAINED = CAPABILITY["text_image_retained_concepts"]
    print("\nretained (all available modalities):", RETAINED)
    print("retained (text+image only):        ", CAPABILITY["text_image_retained_concepts"])
    SCIENTIFIC_PRECONDITIONS = len(RETAINED) >= 2
    if not SCIENTIFIC_PRECONDITIONS:
        print("\nDATA/CAPABILITY NO-GO: fewer than two concepts passed text+image. "
              "Expensive activation and causal stages will be skipped.")

## 10. Load and validate the frozen J-lens

The lens is an existing text-calibrated artifact. It is checksum-pinned,
validated against this model's repo, revision, width, and layers, and then used
read-only. There is no fallback: no random basis, no SAE dictionary, no
freshly-fitted cross-modal lens. A missing or incompatible artifact is a NO-GO
with a named cause.

In [ ]:
# 10. Locate, checksum, and validate the frozen lens.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.lens import JacobianLens
    from jlens.metadata import file_sha256
    from jlens.mmpilot.jspace import CONVENTIONS, validate_lens

    if RUN_REAL_PILOT:
        LENS_PATH = Path(RUNS_ROOT) / LENS_RUN_DIR_NAME / LENS_ARTIFACT_RELPATH
        if not LENS_PATH.is_file():
            raise RuntimeError(
                f"NO-GO: the frozen text-calibrated lens was not found at {LENS_PATH}. "
                "This pilot does not fit a lens. Point LENS_RUN_DIR_NAME at the "
                "completed pilot run directory in Drive."
            )
        LENS_CHECKSUM = file_sha256(str(LENS_PATH))
        LENS = JacobianLens.load(str(LENS_PATH))
    else:
        from jlens.mmpilot.mock import MOCK_LENS_CHECKSUM, mock_lens

        LENS_PATH = RUN_DIR / "mock_lens.pt"
        LENS_CHECKSUM = MOCK_LENS_CHECKSUM
        LENS = mock_lens(CONFIG.layers)

    LENS_VALIDATION = validate_lens(
        LENS,
        lens_path=str(LENS_PATH),
        lens_checksum=LENS_CHECKSUM,
        layers=CONFIG.layers,
        model_repo_id=MODEL_REPO_ID if RUN_REAL_PILOT else "mock/gemma-like",
        model_revision=MODEL_REVISION_USED,
        expect_model_repo_id=MODEL_REPO_ID if RUN_REAL_PILOT else "mock/gemma-like",
        expect_model_revision=MODEL_REVISION_USED,
        expect_d_model=EXPECT_D_MODEL if RUN_REAL_PILOT else BACKEND.d_model,
        expect_checksum=LENS_EXPECT_SHA256 if RUN_REAL_PILOT else MOCK_LENS_CHECKSUM,
    )
    print(json.dumps(LENS_VALIDATION, indent=2, default=str))
    print("\nconventions in force:", json.dumps(CONVENTIONS, indent=2))

## 11. Extract activations

In [ ]:
# 11. Final-prompt-token residual at both layers, one atomic unit per
# (sample, modality, layer). Tensors go to CPU immediately; a resumed run skips
# every checksum-valid unit.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.mmpilot.pipeline import StageOutcome, stage_activations

    if SCIENTIFIC_PRECONDITIONS:
        ACTIVATIONS = stage_activations(
            BACKEND,
            STORE,
            SUBSET,
            CONFIG,
            MEDIA,
            modalities=AVAILABLE_MODALITIES,
            retained_concepts=RETAINED,
            model_revision=MODEL_REVISION_USED,
        )
        print(ACTIVATIONS.line("activations"))
    else:
        ACTIVATIONS = StageOutcome()
        print('activations: skipped (fewer than two capability-passing concepts)')
    _norms = {}
    for record in ACTIVATIONS.records:
        _norms.setdefault((record["modality"], record["layer"]), []).append(record["norm"])
    for key in sorted(_norms):
        values = _norms[key]
        print(f"  {key[0]:13s} L{key[1]}: n={len(values):3d} "
              f"mean norm={sum(values) / len(values):.2f}")

## 12. Compute J-space codes, and check them against matched random directions

There is **no absolute reconstruction threshold**. A sparse workspace is
expected to explain only a small share of an activation: the published J-space
result reports a median of roughly 6-7% of a concept vector's variance in its
top-k J-space component, and excess over a same-size random control that never
exceeds about 10%. A 50% gate — which this notebook previously applied — would
have failed the published result itself.

So absolute explained fraction is reported as a descriptive statistic, and the
lens is gated on the question that can actually come out either way: does it
reconstruct held-out text activations **better than random directions matched
to it** in candidate-pool size, sparsity `k`, atom norms, hidden width, dtype,
device and pursuit settings?

The control's pool is capped so several dense random dictionaries fit next to
the model. Where that cap binds the control searches fewer candidates than the
lens, understating what random could do — a bias in the lens's favour, printed
with every layer.

A short `k` schedule also gives an **occupancy estimate**: the largest `k` at
which the lens's marginal reconstruction gain still beats the control's. It is
inspired by the published occupancy measure and is not a replication of it.

Reconstruction says nothing on its own about causal usefulness. A small-variance
component can still move behaviour, which is what sections 15 and 16 test.

In [ ]:
# 12. Sparse nonnegative decomposition h ~= V s, s >= 0, against the frozen
# dictionary (rows of W_U @ J_l), then the matched-random reconstruction control.
# Dictionaries are built one layer at a time and released, so an L4 can hold a
# bf16 Gemma next to them; the control runs while the layer's dictionary is
# still alive.
#
# There is NO absolute reconstruction threshold. A sparse workspace is expected
# to explain only a small share of an activation — the published J-space result
# reports a median around 6-7% of a concept vector's variance in its top-k
# component — so gating on an absolute level would reject a working lens. The
# criterion is excess over random directions matched to the lens in candidate
# pool, sparsity k, atom norms, width, dtype, device and pursuit settings.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.mmpilot.reconstruction import ReconstructionControlConfig
    from jlens.mmpilot.pipeline import (
        build_dictionaries,
        stage_codes,
        stage_reconstruction_control,
    )
    from jlens.mmpilot.report import code_statistics
    from jlens.mmpilot.reconstruction import summarize_reconstruction_controls

    CONTROL_CONFIG = ReconstructionControlConfig(
        n_draws=5,
        seed=CONFIG.seed,
        max_samples_per_layer=8 if RUN_REAL_PILOT else 4,
        max_control_pool_atoms=16384,
    )
    CODES = []
    CONTROL_RECORDS = []
    DICTIONARIES = {}
    for _layer in (CONFIG.layers if ACTIVATIONS.records else ()):
        DICTIONARIES = build_dictionaries(
            LENS,
            [_layer],
            BACKEND,
            device="cuda" if (RUN_REAL_PILOT and torch.cuda.is_available()) else "cpu",
            dtype=torch.float16 if RUN_REAL_PILOT else torch.float32,
            build_chunk_rows=32768 if RUN_REAL_PILOT else None,
        )
        _outcome = stage_codes(STORE, ACTIVATIONS.records, DICTIONARIES, CONFIG,
                               lens_checksum=LENS_CHECKSUM)
        print(_outcome.line(f"codes L{_layer}"))
        CODES.extend(_outcome.records)
        _control_outcome, _ = stage_reconstruction_control(
            STORE,
            ACTIVATIONS.records,
            DICTIONARIES,
            CONFIG,
            lens_checksum=LENS_CHECKSUM,
            control_config=CONTROL_CONFIG,
            primary_layer=CONFIG.layers[-1],
        )
        print(_control_outcome.line(f"random control L{_layer}"))
        CONTROL_RECORDS.extend(_control_outcome.records)
        if _layer != CONFIG.layers[-1]:
            del DICTIONARIES
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    CODE_STATS = code_statistics(CODES)
    RECONSTRUCTION_CONTROL = summarize_reconstruction_controls(
        CONTROL_RECORDS, config=CONTROL_CONFIG, primary_layer=CONFIG.layers[-1]
    )
    STORE.save("metric", "reconstruction_control", RECONSTRUCTION_CONTROL)

    # Descriptive. Not a gate.
    print()
    print("absolute reconstruction (descriptive only):",
          json.dumps(CODE_STATS, indent=2, default=str))

    print()
    print("held-out text: J-lens vs matched random directions")
    for _layer_key, _entry in sorted(RECONSTRUCTION_CONTROL["by_layer"].items()):
        print(
            f"  L{_entry['layer']}  n={_entry['n_samples']}  "
            f"absolute={_entry['median_explained_fraction']:.4f}  "
            f"random={_entry['median_random_median_explained_fraction']:.4f}  "
            f"excess={_entry['median_excess_explained_fraction']:+.4f}  "
            f"over bound={_entry['median_excess_over_random_bound']:+.4f}  "
            f"occupancy~{_entry['median_estimated_occupancy']}  "
            f"{'ABOVE RANDOM' if _entry['above_random'] else 'not above random'}"
        )
        if _entry.get("control_pool_capped"):
            print(
                f"     note: the control searched {CONTROL_CONFIG.max_control_pool_atoms} "
                f"candidates against the lens's full dictionary, so it understates "
                f"random performance by roughly a factor "
                f"{_entry['max_pool_selection_bias_factor']:.2f} on the greedy "
                f"maximum correlation. The bias favours the lens."
            )

    LENS_SANITY_PASSED = bool(
        SCIENTIFIC_PRECONDITIONS and RECONSTRUCTION_CONTROL["layers_above_random"]
    )
    print(
        f"\nlens sanity (above matched random): "
        f"{'PASS' if LENS_SANITY_PASSED else 'FAIL'} — layers clearing the control: "
        f"{RECONSTRUCTION_CONTROL['layers_above_random']}"
    )
    if not LENS_SANITY_PASSED:
        print(
            "LENS NO-GO: held-out text reconstruction was not distinguishable "
            "from random directions matched to the lens. This is NOT a statement "
            "that the absolute explained fraction was too low — no absolute "
            "threshold is applied. Representational and causal stages are skipped."
        )

## 13. Run representational tests

Cross-modal nearest-neighbour retrieval, matched vs mismatched concept
similarity, coefficient-weighted support overlap, the raw-residual cosine
baseline, and a shuffled-label control — reported per ordered modality pair. A
query never retrieves a group from its own image, so a hit cannot be the
dataset's own caption/recording/image pairing being read back.

Retrieval is representational evidence. It is not causal evidence, and nothing
below treats it as such. No linear probe is fitted: the design brief allows one
only as a cheap diagnostic, and it would not change any decision here.

In [ ]:
# 13. Representational tests at each layer.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.mmpilot.pipeline import stage_representational

    REPRESENTATIONAL = {}
    for _layer in (CONFIG.layers if LENS_SANITY_PASSED else ()):
        REPRESENTATIONAL[_layer] = stage_representational(
            STORE, ACTIVATIONS.records, CODES, CONFIG,
            layer=_layer, modalities=AVAILABLE_MODALITIES,
        )
        print(f"\nlayer {_layer}")
        for pair, entry in sorted(REPRESENTATIONAL[_layer]["pairs"].items()):
            print(
                f"  {pair:28s} jspace top1={entry['jspace_retrieval']['top1_accuracy']:.2f} "
                f"mrr={entry['jspace_retrieval']['mrr']:.2f} | "
                f"raw top1={entry['raw_residual_retrieval']['top1_accuracy']:.2f} | "
                f"shuffled p95={entry['shuffled_control']['p95_top1_accuracy']:.2f} | "
                f"gap={entry['jspace_separation']['gap']:+.3f} "
                f"(raw {entry['raw_residual_separation']['gap']:+.3f}) "
                f"overlap gap={entry['jspace_support_overlap']['gap']:+.3f}"
            )

    PRIMARY_LAYER = CONFIG.layers[-1]
    if not LENS_SANITY_PASSED:
        REPRESENTATIONAL[PRIMARY_LAYER] = {
            'layer': PRIMARY_LAYER, 'pairs': {},
            'skipped_reason': 'held-out text lens reconstruction gate failed',
        }

## 14. Estimate concept directions

In [ ]:
# 14. Directions from SOURCE-modality TRAINING examples only:
#     delta = ReLU(mean positive code - mean matched negative code), top-k kept,
#     v = V delta, normalized to unit L2 (scaled at intervention time by the
#     target modality's mean activation norm). No target-modality data is read.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.mmpilot.pipeline import stage_directions

    if LENS_SANITY_PASSED:
        DIRECTION_OUTCOME, DIRECTIONS = stage_directions(
            STORE, CODES, ACTIVATIONS.records, DICTIONARIES, CONFIG,
            concepts=RETAINED, modalities=AVAILABLE_MODALITIES, lens_checksum=LENS_CHECKSUM,
        )
        print(DIRECTION_OUTCOME.line("directions"))
    else:
        DIRECTION_OUTCOME, DIRECTIONS = StageOutcome(), {}
        print('directions: skipped (lens sanity gate failed)')
    for key, record in sorted(DIRECTIONS.items(), key=lambda kv: [str(x) for x in kv[0]]):
        concept, source_modality, layer, kind = key
        print(f"  {concept:10s} from {source_modality:13s} L{layer} {kind:24s} "
              f"support={len(record.get('support', [])):3d} "
              f"norm={record['reconstructed_norm']:.3f} "
              f"target_data_used={record['uses_target_modality_data']}")

    CAUSAL_CONCEPTS = tuple(RETAINED[:2])
    print("\nconcepts carried into the causal test:", CAUSAL_CONCEPTS)

## 15. Run causal-transfer interventions

For each source x target modality pair: subtract the source-derived direction
from a held-out **positive** target example, add it to a matched held-out
**negative**, and measure the change in complete candidate-sequence scores and
margins. Controls — zero coefficient, a freshly drawn norm-matched random
direction, an unrelated concept's direction, and the raw residual
positive-minus-negative direction — go through the identical code path.

`signed_target_effect` folds in the sign, so positive always means "moved as
intended". This is directional addition and subtraction, not erasure.

In [ ]:
# 15. The source x target transfer matrix with all controls.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.mmpilot.pipeline import stage_causal

    if LENS_SANITY_PASSED and len(RETAINED) >= 2:
        CAUSAL_OUTCOME, INTERVENTIONS = stage_causal(
            BACKEND, STORE, SUBSET, CODES, ACTIVATIONS.records, DIRECTIONS, CONFIG, MEDIA,
            concepts=CAUSAL_CONCEPTS, modalities=AVAILABLE_MODALITIES, all_concepts=RETAINED,
        )
        print(CAUSAL_OUTCOME.line("interventions"))
    else:
        CAUSAL_OUTCOME, INTERVENTIONS = StageOutcome(), {'rows': []}
        print('interventions: skipped (scientific precondition or lens gate failed)')
    print(f"\n{'pair':28s} {'control':24s} {'a':>5s} {'effect':>9s} {'margin':>9s} "
          f"{'sign':>5s} {'unrel':>7s} {'norm':>5s}")
    for row in INTERVENTIONS["rows"]:
        marker = "*" if row["off_diagonal"] and row["control_kind"] == "source_concept" else " "
        print(f"{marker}{row['pair']:27s} {row['control_kind']:24s} {row['alpha']:5.2f} "
              f"{row['mean_signed_target_effect']:+9.4f} {row['mean_signed_margin_effect']:+9.4f} "
              f"{row['fraction_expected_sign']:5.2f} {row['mean_abs_unrelated_change']:7.4f} "
              f"{row['mean_activation_norm_ratio']:5.2f}")
    print("\n* = off-diagonal source-derived direction: the cells that answer the question.")

## 16. Generate the GO/NO-GO report

In [ ]:
# 16. Apply the rubric and write report.md + summary.json into the run directory.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    from jlens.mmpilot.pipeline import stage_report

    REPORT_MD, SUMMARY = stage_report(
        STORE,
        config=CONFIG,
        capability=CAPABILITY,
        lens_validation=LENS_VALIDATION,
        codes=CODES,
        representational=REPRESENTATIONAL[PRIMARY_LAYER],
        interventions=INTERVENTIONS,
        invariance=INVARIANCE,
        reconstruction_control=RECONSTRUCTION_CONTROL,
        blocked_modalities=BLOCKED_MODALITIES,
        manifest_audit=MANIFEST_AUDIT,
    )
    print(REPORT_MD)
    print(f"\nwritten: {RUN_DIR / 'report.md'}")
    print(f"written: {RUN_DIR / 'summary.json'}")
    if not SUMMARY["scientific_evidence"]:
        print("\nThis was a MOCK run. It shows the pipeline works. It is not evidence "
              "about Gemma, and its GO must not be reported as a result.")

## 17. Show resume status

In [ ]:
# 17. What this run did, and what a rerun would skip.
if not RUN_MODEL_STAGES:
    print("skipped: RUN_MODEL_STAGES is False")
else:
    STATUS = STORE.status_report()
    print(json.dumps(STATUS, indent=2, default=str))
    print(
        "\nRerunning this notebook with the same configuration and the same run "
        f"directory ({RUN_DIR}) resumes: every checksum-valid unit above is reused "
        "and only missing work is recomputed. Changing the model revision, the "
        "layers, the lens, the manifest, the split, or the intervention config "
        "changes the fingerprint, and the run directory is then refused rather "
        "than mixed."
    )
    if STATUS["invalid_units"]:
        print("\nunits that failed their checksum and will be recomputed:")
        for path in STATUS["invalid_units"]:
            print("  ", path)

## Interpretation boundary

- SpokenCOCO gives images, written captions, and spoken readings of those
  captions. Results here concern **visual**, **written-linguistic**, and
  **spoken-linguistic** evidence. Nothing here is evidence about environmental
  audio.
- The lens was fitted on text. Applying it to image- and speech-conditioned
  decoder states is the experiment, not an assumption; it is frozen throughout.
- Interventions add and subtract a direction on the residual stream. That is
  not erasure and not projection ablation.
- Retrieval and clustering are representational evidence. Only section 14
  speaks to causation, and only its off-diagonal cells speak to *transfer*.
- A MOCK run proves the pipeline executes. It is never a scientific result.